# LSTM Classifier v4 + SE — Country Embedding + Multi-Task + Macis SE feature

**v4 base 확정** (test PR-AUC 0.0805 / gain +0.045, v1 대비 +12.7%) → 여기에 SE 피처 1개 추가.

## v4_se — SE feature 추가
- Macis AE에서 추출한 `se_score` 컬럼 (`input/processed/features/se_scores.parquet`) join
- 키: (country/iso3, date), per (국가×일) 1개 값
- **log1p 변환 필수**: SE는 분포가 long-tail (median 0.19 / max 74,836, TUR 쿠데타). log1p로 압축해야 LSTM 입력 스케일 안정
- 결측 (2014 초반 + 일부, ~0.8%) → 0으로 채움 (log1p(0)=0, 정상값 가깝게 위치)
- StandardScaler는 log1p 적용 후 fit
- 입력 차원: 55 → **56 features**
- 그 외(architecture, multi-task, embedding, hyperparameter)는 v4 그대로

## 합격 판정 (v4 대비)
- v4_se test PR-AUC > 0.0805 **AND** persistence_gain > +0.045
- 통과 → v4_se가 최종 base
- 미달 → v4 base 유지, Focal Loss 시도

**출력**: `output/lstm_classifier_12y_v4_se/`


## 1. Drive 마운트 + 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/conflict-early-warning'
DATASET_DIR = f'{DRIVE_ROOT}/input/processed/dataset'
SE_PATH = f'{DRIVE_ROOT}/input/processed/features/se_scores.parquet'
OUTPUT_DIR = f'{DRIVE_ROOT}/output/lstm_classifier_12y_v4_se'

# 필수 파일 검증
for p in [f'{DATASET_DIR}/train.parquet', f'{DATASET_DIR}/val.parquet', f'{DATASET_DIR}/test.parquet']:
    assert os.path.isfile(p), f'필수 파일 없음: {p}'
assert os.path.isfile(SE_PATH), f'SE 파일 없음: {SE_PATH} — Drive input/processed/features/ 에 업로드 필요'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Drive OK')
print('  dataset :', DATASET_DIR)
print('  se      :', SE_PATH)
print('  output  :', OUTPUT_DIR)


## 2. 의존성 + GPU 확인

In [ ]:
import sys, time, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, precision_recall_curve

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device :', device)
print('torch  :', torch.__version__)
if device.type == 'cuda':
    print('gpu    :', torch.cuda.get_device_name(0))


## 3. 설정 (split, feature 컬럼, 하이퍼파라미터)

In [ ]:
from datetime import date

SPLIT_DATES = {
    'train_start': pd.Timestamp('2014-01-01', tz='UTC'),
    'train_end':   pd.Timestamp('2023-12-31', tz='UTC'),
    'val_end':     pd.Timestamp('2024-06-30', tz='UTC'),
    'test_end':    pd.Timestamp('2025-03-28', tz='UTC'),
}

TARGET = 'y_escalation'          # 메인 평가 타겟
AUX_TARGETS = ['y_onset', 'y']    # multi-task auxiliary heads

LABEL_META_COLS = [
    'y', 'y_onset', 'y_escalation',
    'fatalities_next3d', 'event_count_next3d',
    'past14d_event_count', 'past14d_fatalities_mean',
]

CONFIG = {
    'seq_len': 30,
    'hidden_dim': 128,
    'dense_dim': 64,
    'dropout': 0.3,
    'country_emb_dim': 8,         # v4 — Country Embedding 차원
    'loss_alpha': 1.0,            # v4 — y_escalation 가중치
    'loss_beta': 0.3,             # v4 — y_onset 보조 가중치
    'loss_gamma': 0.3,            # v4 — y(continuation) 보조 가중치
    'lr': 1e-3,
    'batch_size': 256,
    'epochs': 50,
    'patience': 10,
    'pos_weight_clip': 30.0,
    'pos_weight_onset_clip': 100.0,
}

print('config :', CONFIG)


## 4. 데이터 로드

In [ ]:
t0 = time.time()
train_df = pd.read_parquet(f'{DATASET_DIR}/train.parquet')
val_df   = pd.read_parquet(f'{DATASET_DIR}/val.parquet')
test_df  = pd.read_parquet(f'{DATASET_DIR}/test.parquet')

# 11년치 시퀀스 생성을 위해 full로 합쳤다가 split 시점 기준으로 자름
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
full_df['date'] = pd.to_datetime(full_df['date'], utc=True)
full_df = full_df.sort_values(['country', 'date']).reset_index(drop=True)

# === v4_se: SE feature join (key: country(iso3), date) ===
se_df = pd.read_parquet(SE_PATH)
se_df['date'] = pd.to_datetime(se_df['date'], utc=True)
se_df = se_df.rename(columns={'iso3': 'country'})

before_rows = len(full_df)
full_df = full_df.merge(se_df[['country', 'date', 'se_score']], on=['country', 'date'], how='left')
assert len(full_df) == before_rows, f'merge로 행수 변경: {before_rows} -> {len(full_df)}'

# 결측 처리: 0으로 채움 (log1p(0)=0, 정상 패턴 가깝게)
se_nan_ratio = full_df['se_score'].isna().mean()
full_df['se_score'] = full_df['se_score'].fillna(0.0)

# log1p 변환 (long-tail 압축: max 74836 -> log1p ~ 11)
full_df['se_score'] = np.log1p(full_df['se_score'])
print(f'SE join: NaN ratio (pre-fill) = {se_nan_ratio:.4f}')
print(f'SE after log1p: min={full_df["se_score"].min():.4f} median={full_df["se_score"].median():.4f} max={full_df["se_score"].max():.4f}')
# ============================================================

FEATURE_COLS = [c for c in full_df.columns if c not in ['date', 'country'] + LABEL_META_COLS]
N_FEATURES = len(FEATURE_COLS)

print(f'load: {time.time()-t0:.1f}s')
print(f'  full shape : {full_df.shape}')
print(f'  countries  : {full_df["country"].nunique()}')
print(f'  date range : {full_df["date"].min().date()} ~ {full_df["date"].max().date()}')
print(f'  n_features : {N_FEATURES}  (v4: 55, v4_se: 56)')
print(f'  pos rate (train): {train_df[TARGET].mean():.4f}')


## 5. 국가별 StandardScaler (train 구간으로만 fit)

In [ ]:
scalers = {}
train_mask_full = (full_df['date'] >= SPLIT_DATES['train_start']) & (full_df['date'] <= SPLIT_DATES['train_end'])

for ctry, grp in full_df[train_mask_full].groupby('country'):
    s = StandardScaler()
    s.fit(grp[FEATURE_COLS].values)
    scalers[ctry] = s

# 전체 데이터에 적용 (per-country)
full_df_scaled = full_df.copy()
for ctry, grp in full_df.groupby('country'):
    if ctry in scalers:
        full_df_scaled.loc[grp.index, FEATURE_COLS] = scalers[ctry].transform(grp[FEATURE_COLS].values)
    else:
        # train에 없던 국가 (있을 수 없지만 방어)
        s = StandardScaler()
        s.fit(grp[FEATURE_COLS].values)
        full_df_scaled.loc[grp.index, FEATURE_COLS] = s.transform(grp[FEATURE_COLS].values)

print(f'scalers : {len(scalers)}개국 학습')
print(f'scaled  : NaN {full_df_scaled[FEATURE_COLS].isna().sum().sum()}개')


## 6. 시퀀스 빌더 — 국가별 sliding window

각 (country, t)에 대해 `[t-seq_len+1, t]` 30일 윈도우 → `y_escalation[t]` 예측.
국가 시작 첫 (seq_len-1)일은 윈도우 부족으로 제외.

In [ ]:
SEQ_LEN = CONFIG['seq_len']

# 국가 → 인덱스 매핑 (v4 Country Embedding용)
ALL_COUNTRIES = sorted(full_df_scaled['country'].unique())
COUNTRY_TO_IDX = {c: i for i, c in enumerate(ALL_COUNTRIES)}
N_COUNTRIES = len(ALL_COUNTRIES)
print(f'countries : {N_COUNTRIES}개국')

def build_sequences_v4(df_scaled: pd.DataFrame, date_lo, date_hi):
    """
    v4: 3개 타겟(y_escalation, y_onset, y) + country_idx 함께 반환.
    """
    date_lo_np = np.datetime64(pd.Timestamp(date_lo).tz_convert('UTC').tz_localize(None))
    date_hi_np = np.datetime64(pd.Timestamp(date_hi).tz_convert('UTC').tz_localize(None))

    Xs, Ys, cidxs, metas = [], [], [], []
    for ctry, grp in df_scaled.groupby('country'):
        c_idx = COUNTRY_TO_IDX[ctry]
        feats = grp[FEATURE_COLS].values.astype(np.float32)
        y_esc = grp['y_escalation'].values.astype(np.float32)
        y_ons = grp['y_onset'].values.astype(np.float32)
        y_y   = grp['y'].values.astype(np.float32)
        dates = grp['date'].dt.tz_convert('UTC').dt.tz_localize(None).values
        n = len(grp)
        if n < SEQ_LEN:
            continue
        for t in range(SEQ_LEN - 1, n):
            d = dates[t]
            if d < date_lo_np or d > date_hi_np:
                continue
            Xs.append(feats[t - SEQ_LEN + 1 : t + 1])
            Ys.append([y_esc[t], y_ons[t], y_y[t]])
            cidxs.append(c_idx)
            metas.append((ctry, pd.Timestamp(d, tz='UTC')))
    if not Xs:
        return (
            np.empty((0, SEQ_LEN, N_FEATURES), dtype=np.float32),
            np.empty((0, 3), dtype=np.float32),
            np.empty(0, dtype=np.int64),
            pd.DataFrame(columns=['country','date']),
        )
    X = np.stack(Xs)
    Y = np.array(Ys, dtype=np.float32)
    cidx = np.array(cidxs, dtype=np.int64)
    meta = pd.DataFrame(metas, columns=['country', 'date'])
    return X, Y, cidx, meta

t0 = time.time()
X_train, Y_train, cidx_train, meta_train = build_sequences_v4(full_df_scaled, SPLIT_DATES['train_start'], SPLIT_DATES['train_end'])
X_val,   Y_val,   cidx_val,   meta_val   = build_sequences_v4(full_df_scaled, SPLIT_DATES['train_end'] + pd.Timedelta(days=1), SPLIT_DATES['val_end'])
X_test,  Y_test,  cidx_test,  meta_test  = build_sequences_v4(full_df_scaled, SPLIT_DATES['val_end'] + pd.Timedelta(days=1), SPLIT_DATES['test_end'])

# 평가 편의용으로 y_train/val/test는 escalation만 분리
y_train = Y_train[:, 0]; y_val = Y_val[:, 0]; y_test = Y_test[:, 0]

print(f'seq build: {time.time()-t0:.1f}s')
print(f'  train : X {X_train.shape}  Y {Y_train.shape}  cidx {cidx_train.shape}')
print(f'    y_esc pos : {Y_train[:,0].sum():.0f}/{len(Y_train)} ({Y_train[:,0].mean():.4f})')
print(f'    y_onset pos: {Y_train[:,1].sum():.0f}/{len(Y_train)} ({Y_train[:,1].mean():.4f})')
print(f'    y pos     : {Y_train[:,2].sum():.0f}/{len(Y_train)} ({Y_train[:,2].mean():.4f})')
print(f'  val   : X {X_val.shape}    y_esc pos {Y_val[:,0].sum():.0f}/{len(Y_val)}')
print(f'  test  : X {X_test.shape}   y_esc pos {Y_test[:,0].sum():.0f}/{len(Y_test)}')

mb = (X_train.nbytes + X_val.nbytes + X_test.nbytes) / 1024**2
print(f'  X total : {mb:.0f} MB')


## 7. Dataset / DataLoader

In [ ]:
class SeqDatasetV4(Dataset):
    def __init__(self, X, Y, cidx):
        self.X = torch.from_numpy(X)
        self.Y = torch.from_numpy(Y)     # (N, 3)
        self.cidx = torch.from_numpy(cidx).long()
    def __len__(self):
        return len(self.Y)
    def __getitem__(self, i):
        return self.X[i], self.Y[i], self.cidx[i]

train_loader = DataLoader(SeqDatasetV4(X_train, Y_train, cidx_train), batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(SeqDatasetV4(X_val,   Y_val,   cidx_val),   batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(SeqDatasetV4(X_test,  Y_test,  cidx_test),  batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f'batches train/val/test : {len(train_loader)} / {len(val_loader)} / {len(test_loader)}')


## 8. LSTM Classifier 모델

In [ ]:
class LSTMClassifierV4(nn.Module):
    """
    v4: Country Embedding + Shared LSTM trunk + 3 task heads.
    - 입력 (B, T, F) + country_idx (B,) → (B, T, F+E) for LSTM
    - 출력: (logit_esc, logit_onset, logit_y) 3개
    """
    def __init__(self, n_features, n_countries, country_emb_dim, hidden_dim, dense_dim, dropout):
        super().__init__()
        self.country_emb = nn.Embedding(n_countries, country_emb_dim)
        self.lstm = nn.LSTM(n_features + country_emb_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim, dense_dim)
        self.relu = nn.ReLU()
        # 3 heads (shared trunk)
        self.head_esc = nn.Linear(dense_dim, 1)
        self.head_onset = nn.Linear(dense_dim, 1)
        self.head_y = nn.Linear(dense_dim, 1)

    def forward(self, x, cidx):
        # x: (B, T, F), cidx: (B,)
        emb = self.country_emb(cidx)                          # (B, E)
        emb_seq = emb.unsqueeze(1).expand(-1, x.size(1), -1)  # (B, T, E)
        x_aug = torch.cat([x, emb_seq], dim=-1)               # (B, T, F+E)
        out, _ = self.lstm(x_aug)
        last = out[:, -1, :]
        z = self.dropout(last)
        z = self.relu(self.fc1(z))
        return (
            self.head_esc(z).squeeze(-1),
            self.head_onset(z).squeeze(-1),
            self.head_y(z).squeeze(-1),
        )

model = LSTMClassifierV4(
    n_features=N_FEATURES,
    n_countries=N_COUNTRIES,
    country_emb_dim=CONFIG['country_emb_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    dense_dim=CONFIG['dense_dim'],
    dropout=CONFIG['dropout'],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'params : {n_params:,}')
print(f'  country_emb : {N_COUNTRIES} x {CONFIG["country_emb_dim"]} = {N_COUNTRIES * CONFIG["country_emb_dim"]}')


## 9. 학습 — BCE + pos_weight, early stopping

In [ ]:
# 각 head별 pos_weight 계산
y_esc_train, y_ons_train, y_y_train = Y_train[:,0], Y_train[:,1], Y_train[:,2]

pw_esc = min((1 - y_esc_train.mean()) / max(y_esc_train.mean(), 1e-6), CONFIG['pos_weight_clip'])
pw_ons = min((1 - y_ons_train.mean()) / max(y_ons_train.mean(), 1e-6), CONFIG['pos_weight_onset_clip'])
pw_y   = min((1 - y_y_train.mean())   / max(y_y_train.mean(),   1e-6), CONFIG['pos_weight_clip'])

print(f'pos_weight : esc={pw_esc:.2f}  onset={pw_ons:.2f}  y={pw_y:.2f}')
print(f'pos_rate   : esc={y_esc_train.mean():.4f}  onset={y_ons_train.mean():.4f}  y={y_y_train.mean():.4f}')

criterion_esc   = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw_esc], device=device))
criterion_onset = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw_ons], device=device))
criterion_y     = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw_y], device=device))

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])

ALPHA = CONFIG['loss_alpha']
BETA  = CONFIG['loss_beta']
GAMMA = CONFIG['loss_gamma']

def run_epoch(loader, train_mode):
    if train_mode: model.train()
    else: model.eval()

    total_loss, total_n = 0.0, 0
    probs_esc_all, ys_esc_all = [], []

    for xb, yb, cb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        cb = cb.to(device, non_blocking=True)
        if train_mode: optimizer.zero_grad()

        l_esc, l_ons, l_y = model(xb, cb)
        loss = (
            ALPHA * criterion_esc(l_esc,   yb[:,0]) +
            BETA  * criterion_onset(l_ons, yb[:,1]) +
            GAMMA * criterion_y(l_y,       yb[:,2])
        )

        if train_mode:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * len(yb)
        total_n += len(yb)

        if not train_mode:
            probs_esc_all.append(torch.sigmoid(l_esc).detach().cpu().numpy())
            ys_esc_all.append(yb[:,0].detach().cpu().numpy())

    avg_loss = total_loss / total_n
    if train_mode:
        return avg_loss, None
    probs = np.concatenate(probs_esc_all)
    ys = np.concatenate(ys_esc_all)
    pr_auc = average_precision_score(ys, probs) if ys.sum() > 0 else float('nan')
    return avg_loss, pr_auc

best_val_pr = -1.0
best_state = None
best_epoch = -1
patience_cnt = 0
history = []

t0 = time.time()
for ep in range(1, CONFIG['epochs'] + 1):
    tr_loss, _ = run_epoch(train_loader, train_mode=True)
    val_loss, val_pr = run_epoch(val_loader, train_mode=False)
    history.append({'epoch': ep, 'train_loss': tr_loss, 'val_loss': val_loss, 'val_pr_auc': val_pr})

    improved = val_pr > best_val_pr
    if improved:
        best_val_pr = val_pr
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = ep
        patience_cnt = 0
    else:
        patience_cnt += 1

    flag = '*' if improved else ' '
    print(f'ep {ep:3d} {flag} train {tr_loss:.4f}  val {val_loss:.4f}  val_PR_esc {val_pr:.4f}  ({time.time()-t0:.0f}s)')

    if patience_cnt >= CONFIG['patience']:
        print(f'early stop @ epoch {ep}  (best epoch {best_epoch}, val PR-AUC {best_val_pr:.4f})')
        break

if best_state is not None:
    model.load_state_dict(best_state)

hist_df = pd.DataFrame(history)
hist_df.to_csv(f'{OUTPUT_DIR}/train_history.csv', index=False)
print(f'train done : {time.time()-t0:.0f}s, best epoch {best_epoch}, val PR-AUC {best_val_pr:.4f}')


## 10. 추론 — val/test 확률 산출

In [ ]:
@torch.no_grad()
def predict_loader(loader):
    model.eval()
    probs = []
    for xb, _, cb in loader:
        xb = xb.to(device, non_blocking=True)
        cb = cb.to(device, non_blocking=True)
        l_esc, _, _ = model(xb, cb)
        probs.append(torch.sigmoid(l_esc).cpu().numpy())
    return np.concatenate(probs)

val_prob  = predict_loader(val_loader)
test_prob = predict_loader(test_loader)

pred_val  = meta_val.copy();  pred_val['y_true']  = y_val;  pred_val['y_prob']  = val_prob;  pred_val['split']  = 'val'
pred_test = meta_test.copy(); pred_test['y_true'] = y_test; pred_test['y_prob'] = test_prob; pred_test['split'] = 'test'
predictions = pd.concat([pred_val, pred_test], ignore_index=True)
predictions['date'] = pd.to_datetime(predictions['date'], utc=True)
predictions.to_parquet(f'{OUTPUT_DIR}/predictions.parquet', index=False)
print(f'predictions saved : {len(predictions)} rows')
print(predictions.head())


## 11. 평가 — 6지표군

In [ ]:
def ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.digitize(y_prob, bins[1:-1])
    total = len(y_true)
    e = 0.0
    for b in range(n_bins):
        mask = idx == b
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = y_prob[mask].mean()
        e += (mask.sum() / total) * abs(acc - conf)
    return float(e)

def top_k_metrics(y_true, y_prob, k_pct):
    k = max(1, int(len(y_prob) * k_pct / 100))
    idx = np.argsort(-y_prob)[:k]
    prec = y_true[idx].mean()
    rec = y_true[idx].sum() / max(y_true.sum(), 1)
    return float(prec), float(rec)

def recall_at_p(y_true, y_prob, p_target):
    p, r, _ = precision_recall_curve(y_true, y_prob)
    mask = p >= p_target
    if mask.sum() == 0:
        return float('nan')
    return float(r[mask].max())

# Persistence baseline: past14d_event_count (test 시점)
# meta_test의 (country, date) 기준으로 full_df에서 끌어옴
ref = full_df[['country', 'date', 'past14d_event_count']].copy()
ref['date'] = pd.to_datetime(ref['date'], utc=True)
mt = meta_test.copy()
mt['date'] = pd.to_datetime(mt['date'], utc=True)
mt = mt.merge(ref, on=['country', 'date'], how='left')
persistence_score = mt['past14d_event_count'].fillna(0).values
pr_auc_persistence = average_precision_score(y_test, persistence_score) if y_test.sum() > 0 else float('nan')

pr_auc = average_precision_score(y_test, test_prob)
persistence_gain = pr_auc - pr_auc_persistence

p1, r1 = top_k_metrics(y_test, test_prob, 1)
p5, r5 = top_k_metrics(y_test, test_prob, 5)
p10, r10 = top_k_metrics(y_test, test_prob, 10)

# Macro PR-AUC (국가별 평균, 양성≥1 국가만)
macro_aucs = []
for ctry, grp in meta_test.assign(y_true=y_test, y_prob=test_prob).groupby('country'):
    if grp['y_true'].sum() > 0:
        macro_aucs.append(average_precision_score(grp['y_true'], grp['y_prob']))
macro_pr = float(np.mean(macro_aucs)) if macro_aucs else float('nan')

# Lead time: val 95퍼센타일 임계 τ
tau = float(np.percentile(val_prob, 95))
# (test에서 양성 hit 시점이 실제 분쟁일까지 며칠 앞섰는지 — 라벨 정의상 단순화: τ 초과 첫날에서 양성일 사이 거리)
mt2 = meta_test.assign(y_true=y_test, y_prob=test_prob).sort_values(['country','date'])
lead_times = []
for ctry, grp in mt2.groupby('country'):
    grp = grp.reset_index(drop=True)
    pos_dates = grp.loc[grp['y_true'] == 1, 'date'].tolist()
    alert_dates = grp.loc[grp['y_prob'] >= tau, 'date'].tolist()
    for pd_ in pos_dates:
        earlier = [a for a in alert_dates if a <= pd_]
        if earlier:
            delta = (pd_ - max(earlier)).days
            lead_times.append(delta)
lead_median = float(np.median(lead_times)) if lead_times else float('nan')

eval_result = {
    'pr_auc': float(pr_auc),
    'pr_auc_macro': macro_pr,
    'n_countries_macro': len(macro_aucs),
    'pr_auc_persistence': float(pr_auc_persistence),
    'persistence_gain': float(persistence_gain),
    'precision_top1pct': p1, 'recall_top1pct': r1,
    'precision_top5pct': p5, 'recall_top5pct': r5,
    'precision_top10pct': p10, 'recall_top10pct': r10,
    'ece': ece(y_test, test_prob),
    'lead_time_median_days': lead_median,
    'lead_time_threshold': tau,
    'recall_at_p10': recall_at_p(y_test, test_prob, 0.10),
    'recall_at_p20': recall_at_p(y_test, test_prob, 0.20),
    'recall_at_p30': recall_at_p(y_test, test_prob, 0.30),
    'best_epoch': best_epoch,
    'best_val_pr_auc': float(best_val_pr),
}
print(json.dumps(eval_result, indent=2))
with open(f'{OUTPUT_DIR}/eval.json', 'w') as f:
    json.dump(eval_result, f, indent=2)


## 12. 백테스트 3케이스 — UKR 2022-02-24 / SDN 2023-04-15 / PSE 2023-10-07

각 사건 D-30 ~ D+7 구간 확률 vs 라벨 시각화.

In [ ]:
import matplotlib.pyplot as plt

CASES = [
    ('UKR', pd.Timestamp('2022-02-24', tz='UTC'), 'Ukraine invasion'),
    ('SDN', pd.Timestamp('2023-04-15', tz='UTC'), 'Sudan civil war'),
    ('PSE', pd.Timestamp('2023-10-07', tz='UTC'), 'Gaza war'),
]

def predict_country_full(ctry):
    grp = full_df_scaled[full_df_scaled['country'] == ctry].sort_values('date').reset_index(drop=True)
    feats = grp[FEATURE_COLS].values.astype(np.float32)
    n = len(grp)
    if n < SEQ_LEN:
        return None
    c_idx = COUNTRY_TO_IDX[ctry]
    Xs = np.stack([feats[t - SEQ_LEN + 1 : t + 1] for t in range(SEQ_LEN - 1, n)])
    with torch.no_grad():
        model.eval()
        probs = []
        for i in range(0, len(Xs), 512):
            xb = torch.from_numpy(Xs[i:i+512]).to(device)
            cb = torch.full((len(xb),), c_idx, dtype=torch.long, device=device)
            l_esc, _, _ = model(xb, cb)
            probs.append(torch.sigmoid(l_esc).cpu().numpy())
        probs = np.concatenate(probs)
    out = grp.iloc[SEQ_LEN - 1:].copy()
    out['y_prob'] = probs
    return out

fig, axes = plt.subplots(3, 1, figsize=(11, 9))
for ax, (ctry, ev_date, title) in zip(axes, CASES):
    pred_full = predict_country_full(ctry)
    if pred_full is None:
        ax.set_title(f'{ctry}: data insufficient')
        continue
    lo, hi = ev_date - pd.Timedelta(days=30), ev_date + pd.Timedelta(days=7)
    sl = pred_full[(pred_full['date'] >= lo) & (pred_full['date'] <= hi)]
    ax.plot(sl['date'], sl['y_prob'], label='y_prob', color='tab:blue')
    ax2 = ax.twinx()
    ax2.bar(sl['date'], sl[TARGET], alpha=0.3, color='tab:red', label='y_escalation')
    ax.axvline(ev_date, color='black', linestyle='--', linewidth=1, label='event')
    ax.set_title(f'{ctry} — {title}  ({ev_date.date()})')
    ax.set_ylabel('y_prob'); ax2.set_ylabel('y_esc'); ax.set_ylim(0, 1); ax2.set_ylim(0, 1.05)
    ax.legend(loc='upper left'); ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/backtest_3cases.png', dpi=120, bbox_inches='tight')
plt.show()
print('backtest plot saved')


## 13. 모델 + config 저장

In [ ]:
torch.save({
    'state_dict': model.state_dict(),
    'config': CONFIG,
    'feature_cols': FEATURE_COLS,
    'country_to_idx': COUNTRY_TO_IDX,
    'pos_weight': {'esc': float(pw_esc), 'onset': float(pw_ons), 'y': float(pw_y)},
    'best_epoch': best_epoch,
    'best_val_pr_auc': float(best_val_pr),
}, f'{OUTPUT_DIR}/model.pt')

def _to_native(o):
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    raise TypeError(f'{type(o).__name__} not serializable')

with open(f'{OUTPUT_DIR}/config.json', 'w') as f:
    json.dump({
        **CONFIG,
        'target': TARGET,
        'aux_targets': AUX_TARGETS,
        'n_features': N_FEATURES,
        'n_countries': N_COUNTRIES,
        'feature_cols': FEATURE_COLS,
        'pos_weight': {'esc': float(pw_esc), 'onset': float(pw_ons), 'y': float(pw_y)},
        'split_dates': {k: str(v.date()) for k, v in SPLIT_DATES.items()},
        'eval': eval_result,
    }, f, indent=2, ensure_ascii=False, default=_to_native)

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(hist_df['epoch'], hist_df['train_loss'], label='train')
ax.plot(hist_df['epoch'], hist_df['val_loss'], label='val')
ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.legend()
ax2 = ax.twinx()
ax2.plot(hist_df['epoch'], hist_df['val_pr_auc'], color='tab:green', label='val PR-AUC (esc)')
ax2.set_ylabel('val PR-AUC')
plt.title('LSTM Classifier v4_se — training')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/train_loss.png', dpi=120, bbox_inches='tight')
plt.show()
print('model.pt + config.json + loss curve 저장 완료')


## 14. 결과 파일 목록

In [ ]:
for f in sorted(os.listdir(OUTPUT_DIR)):
    p = f'{OUTPUT_DIR}/{f}'
    sz = os.path.getsize(p) / 1024
    print(f'  {f:30s}  {sz:8.1f} KB')
print()
print('Drive 경로 :', OUTPUT_DIR)
print('완료. 로컬에서 README §4 절차대로 배치.')
